<a href="https://colab.research.google.com/github/shefiramarizcha62-sudo/flyrank-ml-portfolio/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shefiramarizcha62-sudo/flyrank-ml-portfolio/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import duckdb
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

HF_TOKEN = userdata.get("FlyRank-ML")

print("HF token loaded:", HF_TOKEN is not None)

con = duckdb.connect()

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

print("Hugging Face authentication configured.")

HF token loaded: True
Hugging Face authentication configured.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


The Week-5 Random Forest was evaluated using a client-grouped split, with 44 clients in the training set and 11 different clients in the test set. There was no client overlap between the two sets.

For this validation audit, I compare that honest grouped evaluation with a random row-level split. The random split is included as a diagnostic comparison rather than as the preferred estimate, because content rows from the same client can appear in both training and test data.

The grouped split is treated as the more honest evaluation for the capstone decision because it tests whether the model can generalize across clients rather than relying on client-specific patterns.

The comparison uses the same Random Forest approach and the same target definition (`future_opportunity`). Performance is reported using ROC-AUC and Average Precision.

In [5]:
# ==================================================
# SECTION 2 — MY MODEL UNDER AN HONEST SPLIT
# BEFORE / AFTER VALIDATION
# ==================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score


# ==================================================
# 1. LOAD MARCH FEATURE WINDOW
# ==================================================
# March represents the information available at the
# decision moment.

feature_frame = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_engaged_sessions

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    )

    WHERE month = '2026-03'
""").df()


print("March feature rows:", len(feature_frame))

print(
    "March date range:",
    feature_frame["report_date"].min(),
    "to",
    feature_frame["report_date"].max()
)


# ==================================================
# 2. AGGREGATE MARCH FEATURES
# ==================================================
# One row per client-content pair.

march_features = (
    feature_frame
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_avg_position=("gsc_avg_position", "mean"),
        ga4_pageviews=("ga4_pageviews", "sum"),
        ga4_engaged_sessions=("ga4_engaged_sessions", "sum")
    )
)


# Calculate March CTR

march_features["march_ctr"] = (
    march_features["gsc_clicks"]
    / march_features["gsc_impressions"].replace(0, np.nan)
).fillna(0)


print(
    "\nMarch client-content pairs:",
    len(march_features)
)


# ==================================================
# 3. LOAD APRIL OUTCOME WINDOW
# ==================================================
# April is used ONLY to define the future outcome.
# It must not become a model feature.

april_outcome = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks

    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet'
    )

    WHERE month = '2026-04'
""").df()


print("\nApril outcome rows:", len(april_outcome))

print(
    "April date range:",
    april_outcome["report_date"].min(),
    "to",
    april_outcome["report_date"].max()
)


# ==================================================
# 4. AGGREGATE APRIL OUTCOME
# ==================================================

april_outcome = (
    april_outcome
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_clicks=("gsc_clicks", "sum")
    )
)


# ==================================================
# 5. CALCULATE APRIL CTR
# ==================================================

april_outcome["april_ctr"] = (
    april_outcome["april_clicks"]
    / april_outcome["april_impressions"].replace(0, np.nan)
).fillna(0)


# ==================================================
# 6. JOIN MARCH FEATURES WITH APRIL OUTCOME
# ==================================================

model_df = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)


print("\nModeling rows:", len(model_df))

print(
    "Unique client-content pairs:",
    model_df[
        ["client_hash_id", "content_hash_id"]
    ].drop_duplicates().shape[0]
)


# ==================================================
# 7. DEFINE FUTURE OPPORTUNITY LABEL
# ==================================================
# The label is based on April:
# - more than 100 impressions
# - CTR below 0.1%
#
# April is used ONLY for the target.
# No April column is included in X.

model_df["future_opportunity"] = (
    (model_df["april_impressions"] > 100)
    & (model_df["april_ctr"] < 0.001)
).astype(int)


print("\nFuture opportunity distribution:")
print(
    model_df["future_opportunity"]
    .value_counts()
    .sort_index()
)


# ==================================================
# 8. DEFINE MODEL FEATURES
# ==================================================
# These are the same six decision-moment features
# used in ML-08.

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "march_ctr",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

X = model_df[feature_columns].copy()
y = model_df["future_opportunity"].copy()

print("\nFeature columns:")
print(feature_columns)

print("\nX shape:", X.shape)
print("y shape:", y.shape)


# ==================================================
# 9. HANDLE MISSING VALUES
# ==================================================

imputer = SimpleImputer(strategy="median")

X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
    index=X.index
)


# ==================================================
# 10. BEFORE — RANDOM SPLIT
# ==================================================
# This represents the less strict validation design.
# Rows are randomly divided, so the same client may
# appear in both train and test.

X_train_before, X_test_before, y_train_before, y_test_before = (
    train_test_split(
        X_imputed,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)


print("\n==========================================")
print("BEFORE — RANDOM SPLIT")
print("==========================================")

print("Train rows:", len(X_train_before))
print("Test rows:", len(X_test_before))

print(
    "Train target distribution:",
    y_train_before.value_counts(normalize=True)
    .round(4)
    .to_dict()
)

print(
    "Test target distribution:",
    y_test_before.value_counts(normalize=True)
    .round(4)
    .to_dict()
)


# Check client overlap in the random split

before_train_clients = model_df.loc[
    X_train_before.index,
    "client_hash_id"
]

before_test_clients = model_df.loc[
    X_test_before.index,
    "client_hash_id"
]

before_overlap = len(
    set(before_train_clients)
    .intersection(set(before_test_clients))
)

print("Client overlap:", before_overlap)


# ==================================================
# 11. AFTER — GROUPED SPLIT BY CLIENT
# ==================================================
# This is the honest validation improvement.
# A client is present in either train OR test,
# never both.

groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X_imputed,
        y,
        groups=groups
    )
)


X_train_after = X_imputed.iloc[train_idx]
X_test_after = X_imputed.iloc[test_idx]

y_train_after = y.iloc[train_idx]
y_test_after = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]


print("\n==========================================")
print("AFTER — GROUPED BY CLIENT")
print("==========================================")

print("Train rows:", len(X_train_after))
print("Test rows:", len(X_test_after))

print(
    "Train clients:",
    groups_train.nunique()
)

print(
    "Test clients:",
    groups_test.nunique()
)

client_overlap = len(
    set(groups_train)
    .intersection(set(groups_test))
)

print(
    "Client overlap:",
    client_overlap
)

print(
    "Train target distribution:",
    y_train_after.value_counts(normalize=True)
    .round(4)
    .to_dict()
)

print(
    "Test target distribution:",
    y_test_after.value_counts(normalize=True)
    .round(4)
    .to_dict()
)


# ==================================================
# 12. TRAIN RANDOM FOREST — BEFORE
# ==================================================

rf_before = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_before.fit(
    X_train_before,
    y_train_before
)


before_score = rf_before.predict_proba(
    X_test_before
)[:, 1]


# ==================================================
# 13. TRAIN RANDOM FOREST — AFTER
# ==================================================

rf_after = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_after.fit(
    X_train_after,
    y_train_after
)


after_score = rf_after.predict_proba(
    X_test_after
)[:, 1]


print("\nRandom Forest models trained.")
print("Number of trees:", 200)
print("Max depth:", 12)


# ==================================================
# 14. EVALUATE BEFORE
# ==================================================

before_auc = roc_auc_score(
    y_test_before,
    before_score
)

before_ap = average_precision_score(
    y_test_before,
    before_score
)


# ==================================================
# 15. EVALUATE AFTER
# ==================================================

after_auc = roc_auc_score(
    y_test_after,
    after_score
)

after_ap = average_precision_score(
    y_test_after,
    after_score
)


# ==================================================
# 16. BEFORE / AFTER COMPARISON
# ==================================================

validation_comparison = pd.DataFrame({
    "validation_design": [
        "BEFORE — Random split",
        "AFTER — Grouped by client"
    ],
    "ROC_AUC": [
        before_auc,
        after_auc
    ],
    "Average_Precision": [
        before_ap,
        after_ap
    ]
})


print("\n==========================================")
print("BEFORE / AFTER VALIDATION")
print("==========================================")

validation_comparison

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March feature rows: 9841378
March date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00

March client-content pairs: 331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


April outcome rows: 10424730
April date range: 2026-04-01 00:00:00 to 2026-04-30 00:00:00

Modeling rows: 331436
Unique client-content pairs: 331436

Future opportunity distribution:
future_opportunity
0    280002
1     51434
Name: count, dtype: int64

Feature columns:
['gsc_impressions', 'gsc_clicks', 'march_ctr', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']

X shape: (331436, 6)
y shape: (331436,)

BEFORE — RANDOM SPLIT
Train rows: 265148
Test rows: 66288
Train target distribution: {0: 0.8448, 1: 0.1552}
Test target distribution: {0: 0.8448, 1: 0.1552}
Client overlap: 55

AFTER — GROUPED BY CLIENT
Train rows: 300879
Test rows: 30557
Train clients: 44
Test clients: 11
Client overlap: 0
Train target distribution: {0: 0.8467, 1: 0.1533}
Test target distribution: {0: 0.8262, 1: 0.1738}

Random Forest models trained.
Number of trees: 200
Max depth: 12

BEFORE / AFTER VALIDATION


,validation_design,ROC_AUC,Average_Precision
0,BEFORE — Random split,0.914363,0.663663
1,AFTER — Grouped by client,0.921380,0.679493


### Interpretation

The grouped-by-client validation was used as a stricter evaluation design because
content from the same client does not appear in both the training and test sets.

Under the random split, the model achieved a ROC-AUC of 0.914363 and an Average
Precision of 0.663663, with client overlap between training and test data.

After grouping by client, there was no client overlap between training and test
sets. The model achieved a ROC-AUC of 0.921380 and an Average Precision of 0.679493.

In this experiment, the grouped validation scores were slightly higher than the
random-split scores. This is an observed result rather than evidence that grouped
validation universally improves model performance. The grouped split provides a
more conservative test of whether the model can rank future opportunities for
clients not represented in the training set.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
# ==================================================
# SECTION 3 — LEAKAGE AUDIT
# ==================================================

# Final features used by the Random Forest
print("Final model features:")
print(feature_columns)


# --------------------------------------------------
# 1. Check for future/outcome-related columns
# --------------------------------------------------

leakage_keywords = [
    "april",
    "future",
    "outcome",
    "label",
    "target",
    "baseline",
    "product",
    "flag"
]

potential_leakage = [
    col for col in feature_columns
    if any(keyword in col.lower() for keyword in leakage_keywords)
]


print("\nPotential leakage-related columns:")
print(potential_leakage)


# --------------------------------------------------
# 2. Check that target is not included
# --------------------------------------------------

target_in_features = "future_opportunity" in feature_columns

print("\nTarget included in features:")
print(target_in_features)


# --------------------------------------------------
# 3. Check that April outcome columns are excluded
# --------------------------------------------------

future_columns = [
    "april_impressions",
    "april_clicks",
    "april_ctr",
    "future_opportunity"
]

future_in_features = [
    col for col in future_columns
    if col in feature_columns
]

print("\nFuture/outcome columns included in features:")
print(future_in_features)


# --------------------------------------------------
# 4. Final leakage verdict
# --------------------------------------------------

if (
    len(potential_leakage) == 0
    and not target_in_features
    and len(future_in_features) == 0
):
    print("\nLeakage audit: PASS")
    print(
        "The final feature set contains only "
        "decision-moment March signals."
    )
else:
    print("\nLeakage audit: REVIEW REQUIRED")

Final model features:
['gsc_impressions', 'gsc_clicks', 'march_ctr', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']

Potential leakage-related columns:
[]

Target included in features:
False

Future/outcome columns included in features:
[]

Leakage audit: PASS
The final feature set contains only decision-moment March signals.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original claim

The Random Forest model outperformed the ML-07 baseline and can identify future content opportunities more effectively.

### Revised claim

In this experiment, the Random Forest model showed higher measured ROC-AUC and
Average Precision than the ML-07 baseline on the evaluation set. Under the
grouped-by-client validation, the model achieved a ROC-AUC of 0.921380 and an
Average Precision of 0.679493, compared with 0.738707 and 0.318966 for the
ML-07 baseline.

These results are observed and measured on the available FlyRank internship
dataset and evaluation design. They provide directional evidence that the
model may be useful for ranking content for future-opportunity review, but they
do not establish causal effects or guarantee performance on unseen datasets or
future periods. The model should therefore be treated as a decision-support
tool rather than proof of search-ranking or CTR outcomes.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.